Description :

This notebook implements the complete Retrieval-Augmented Generation (RAG) pipeline. It retrieves the most relevant document chunks from the FAISS vector store based on a user query and generates a grounded answer using a large language model constrained by retrieved context.

Key tasks performed:

Encode user queries into embeddings

Retrieve top-k relevant document chunks

Construct context-aware prompts

Generate factual answers using an LLM

Prevent hallucination by grounding responses in official rules

In [1]:
%pip install -q faiss-cpu sentence-transformers transformers torch

Note: you may need to restart the kernel to use updated packages.


In [1]:
import faiss
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [2]:
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

In [3]:
index = faiss.read_index("Data/railway_faiss.index")
print("FAISS index loaded. Total vectors:", index.ntotal)

FAISS index loaded. Total vectors: 23


In [5]:
with open("Data/railway_metadata.json", "r", encoding="utf-8") as f:
    metadata = json.load(f)

print("Total chunks loaded:", len(metadata))


Total chunks loaded: 23


In [6]:
def retrieve_context(query, top_k=5):
    query_embedding = embed_model.encode([query])
    query_embedding = np.array(query_embedding).astype("float32")

    distances, indices = index.search(query_embedding, top_k)

    contexts = []
    for idx in indices[0]:
        contexts.append(metadata[idx]["text"])

    return "\n\n".join(contexts)

In [7]:
tokenizer = AutoTokenizer.from_pretrained("google/flan-t5-base")
model = AutoModelForSeq2SeqLM.from_pretrained("google/flan-t5-base")

In [8]:
def generate_answer(query):
    context = retrieve_context(query)

    prompt = f"""
You are an assistant answering questions strictly based on Indian Railways rules.

Context:
{context}

Question:
{query}

Answer in a factual and clear manner:
"""

    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = model.generate(**inputs, max_new_tokens=200)

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [9]:
print(generate_answer("What are the refund rules if a train is cancelled?"))

The ticket is surrendered within three hours from the actual departure of the train.


In [10]:
print(generate_answer("What facilities are passengers entitled to?"))

TTE, Guard or GRP escort for registration of the report at the next Police Station.


# RAG Evaluation

In [11]:
evaluation_questions = [
    "What are the rights of passengers if a train is cancelled?",
    "How can railway passengers file complaints?",
    "Is compensation provided for train delays?",
    "What ID proof is required during train travel?",
    "What facilities must railways provide to passengers?",
    "Are senior citizens eligible for any concessions?",
    "Can passengers get refund for cancelled tickets?",
    "What are passenger rights related to cleanliness?",
]

In [14]:
rresults = []

for q in evaluation_questions:
    # Unpack only the first two values, ignore extras if any
    answer, contexts, *rest = generate_answer(q)

    results.append({
        "question": q,
        "answer": answer,
        "retrieved_context": contexts[:2]  # top 2 chunks
    })

In [15]:
for r in results:
    print("QUESTION:", r["question"])
    print("ANSWER:", r["answer"])
    print("RETRIEVED CONTEXT:")
    for c in r["retrieved_context"]:
        print("-", c[:300])
    print("-" * 80)

QUESTION: What are the rights of passengers if a train is cancelled?
ANSWER: k
RETRIEVED CONTEXT:
- i
--------------------------------------------------------------------------------
QUESTION: How can railway passengers file complaints?
ANSWER: 1
RETRIEVED CONTEXT:
- .
--------------------------------------------------------------------------------
QUESTION: Is compensation provided for train delays?
ANSWER: i
RETRIEVED CONTEXT:
- i
--------------------------------------------------------------------------------
QUESTION: What ID proof is required during train travel?
ANSWER: T
RETRIEVED CONTEXT:
- T
--------------------------------------------------------------------------------
QUESTION: What facilities must railways provide to passengers?
ANSWER: T
RETRIEVED CONTEXT:
- T
--------------------------------------------------------------------------------
QUESTION: Are senior citizens eligible for any concessions?
ANSWER: *
RETRIEVED CONTEXT:
-  
-----------------------------------------